In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
import matplotlib.pyplot as plt
import seaborn as sns

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [ ]:
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

In [ ]:
train = pd.read_csv("/content/train.csv")
test = pd.read_csv("/content/test.csv")

print(train.head())
print(train.shape)
print(train.columns)

In [ ]:
print(train.isnull().sum())

print("Duplicate rows:", train.duplicated().sum())

train = train.drop_duplicates()

train['comment_text'] = train['comment_text'].fillna('')

In [ ]:
stop_words = set(stopwords.words('english'))

lemmatizer = WordNetLemmatizer()

def clean_text(text):

    text = text.lower()

    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)

    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)

    # Remove special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # Tokenization
    words = text.split()

    # Stopword removal and lemmatization
    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words
    ]

    return ' '.join(words)

In [ ]:
train['cleaned_text'] = train['comment_text'].apply(clean_text)

test['cleaned_text'] = test['comment_text'].apply(clean_text)

print(train[['comment_text', 'cleaned_text']].head())

In [ ]:
target_columns = [
    'toxic',
    'severe_toxic',
    'obscene',
    'threat',
    'insult',
    'identity_hate'
]

class_counts = train[target_columns].sum()

print(class_counts)

plt.figure(figsize=(10, 5))

sns.barplot(
    x=class_counts.index,
    y=class_counts.values
)

plt.title("Toxic Comment Class Distribution")
plt.xlabel("Toxicity Categories")
plt.ylabel("Number of Comments")

plt.xticks(rotation=45)
plt.show()

In [ ]:
train.to_csv("train_processed.csv", index=False)
test.to_csv("test_processed.csv", index=False)

**EDA**

In [ ]:
train = pd.read_csv("train_processed.csv")

print("Dataset Shape:", train.shape)
print(train.head())

In [ ]:
target_columns = [
    'toxic',
    'severe_toxic',
    'obscene',
    'threat',
    'insult',
    'identity_hate'
]

class_counts = train[target_columns].sum()

print(class_counts)

In [ ]:
plt.figure(figsize=(10, 5))

sns.barplot(
    x=class_counts.index,
    y=class_counts.values
)

plt.title("Distribution of Toxic Comments")
plt.xlabel("Toxicity Categories")
plt.ylabel("Number of Comments")
plt.xticks(rotation=45)

plt.show()

In [ ]:
train['is_toxic'] = train[target_columns].max(axis=1)

print(train['is_toxic'].value_counts())

In [ ]:
sns.countplot(
    data=train,
    x='is_toxic'
)

plt.title("Toxic vs Non-Toxic Comments")
plt.xticks([0, 1], ['Non-Toxic', 'Toxic'])
plt.show()

In [ ]:
train['comment_length'] = train['comment_text'].apply(
    lambda x: len(str(x).split())
)

plt.figure(figsize=(10, 5))

sns.histplot(
    data=train,
    x='comment_length',
    hue='is_toxic',
    bins=50,
    element='step'
)

plt.title("Comment Length Distribution")
plt.xlabel("Number of Words")
plt.show()

In [ ]:
# from collections import Counter

# toxic_comments = train[
#     train['is_toxic'] == 1
# ]['cleaned_text']

# non_toxic_comments = train[
#     train['is_toxic'] == 0
# ]['cleaned_text']

# toxic_words = Counter(
#     ' '.join(toxic_comments).split()
# )

# non_toxic_words = Counter(' '.join(non_toxic_comments).split())

# print("Top Toxic Words:", toxic_words.most_common(15))
# print("Top Non-Toxic Words:", non_toxic_words.most_common(15))

In [ ]:
print(train['cleaned_text'].isnull().sum())

In [ ]:
train['cleaned_text'] = train['cleaned_text'].fillna('')
train['cleaned_text'] = train['cleaned_text'].astype(str)
train['comment_text'] = train['comment_text'].fillna('')
train['cleaned_text'] = train['comment_text'].apply(clean_text)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

X = tfidf.fit_transform(train['cleaned_text'])

print("TF-IDF Shape:", X.shape)

In [ ]:
from sklearn.model_selection import train_test_split

target_columns = [
    'toxic',
    'severe_toxic',
    'obscene',
    'threat',
    'insult',
    'identity_hate'
]

y = train[target_columns]

X_train, X_val, y_train, y_val = train_test_split(
    train['cleaned_text'],
    y,
    test_size=0.2,
    random_state=42
)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)

In [ ]:
X_train_tfidf = tfidf.fit_transform(X_train)

X_val_tfidf = tfidf.transform(X_val)

print("Training TF-IDF:", X_train_tfidf.shape)
print("Validation TF-IDF:", X_val_tfidf.shape)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier

model = OneVsRestClassifier(
    LogisticRegression(max_iter=1000)
)

model.fit(X_train_tfidf, y_train)

print("Model training completed!")

In [ ]:
y_pred = model.predict(X_val_tfidf)

print(y_pred[:5])

In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_val,
        y_pred,
        target_names=target_columns,
        zero_division=0
    )
)

In [ ]:
from sklearn.metrics import f1_score

micro_f1 = f1_score(
    y_val,
    y_pred,
    average='micro'
)

macro_f1 = f1_score(
    y_val,
    y_pred,
    average='macro'
)

print("Micro F1:", micro_f1)
print("Macro F1:", macro_f1)

In [ ]:
import pickle

with open("tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf, f)

with open("logistic_model.pkl", "wb") as f:
    pickle.dump(model, f)

print("Models saved successfully!")

**LSTM Model**

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    f1_score,
    roc_auc_score
)

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding,
    LSTM,
    GRU,
    Bidirectional,
    Dense,
    Dropout
)

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)

In [ ]:
train = pd.read_csv("train_processed.csv")

target_columns = [
    'toxic',
    'severe_toxic',
    'obscene',
    'threat',
    'insult',
    'identity_hate'
]

train['comment_text'] = (
    train['comment_text']
    .fillna('')
    .astype(str)
)

X = train['comment_text'].values
y = train[target_columns].values.astype('float32')

print("X shape:", X.shape)
print("Y shape:", y.shape)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))

In [ ]:
max_words = 50000
max_len = 200

tokenizer = Tokenizer(
    num_words=max_words,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)

X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=max_len,
    padding='post',
    truncating='post'
)

X_val_pad = pad_sequences(
    X_val_seq,
    maxlen=max_len,
    padding='post',
    truncating='post'
)

print("Training shape:", X_train_pad.shape)
print("Validation shape:", X_val_pad.shape)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    Embedding,
    LSTM,
    Dense,
    Dropout
)

lstm_model = Sequential([

    Input(shape=(max_len,)),

    Embedding(
        input_dim=max_words,
        output_dim=128,
        mask_zero=True
    ),

    LSTM(
        128,
        return_sequences=False
    ),

    Dropout(0.3),

    Dense(64, activation='relu'),

    Dropout(0.3),

    Dense(6, activation='sigmoid')
])

lstm_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['binary_accuracy']
)

lstm_model.summary()

In [ ]:
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True
    ),

    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=0.00001
    ),

    ModelCheckpoint(
        'best_lstm.keras',
        monitor='val_loss',
        save_best_only=True
    )
]

In [ ]:
history_lstm = lstm_model.fit(
    X_train_pad,
    y_train,
    validation_data=(X_val_pad, y_val),
    epochs=10,
    batch_size=128,
    callbacks=callbacks
)

Epoch 1/10
998/998 ━━━━━━━━━━━━━━━━━━━━ 727s 723ms/step - binary_accuracy: 0.9721 - loss: 0.0920 - val_binary_accuracy: 0.9805 - val_loss: 0.0551 - learning_rate: 0.0010
Epoch 2/10
998/998 ━━━━━━━━━━━━━━━━━━━━ 798s 780ms/step - binary_accuracy: 0.9812 - loss: 0.0527 - val_binary_accuracy: 0.9816 - val_loss: 0.0497 - learning_rate: 0.0010
Epoch 3/10
366/998 ━━━━━━━━━━━━━━━━━━━━ 7:11 683ms/step - binary_accuracy: 0.9833 - loss: 0.0446

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history_lstm.history['loss'], label='Training Loss')
plt.plot(history_lstm.history['val_loss'], label='Validation Loss')
plt.title("LSTM Loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history_lstm.history['binary_accuracy'], label='Training Accuracy')
plt.plot(history_lstm.history['val_binary_accuracy'], label='Validation Accuracy')
plt.title("LSTM Accuracy")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
y_prob_lstm = lstm_model.predict(X_val_pad)

y_pred_lstm = (y_prob_lstm >= 0.5).astype(int)

print(classification_report(
    y_val,
    y_pred_lstm,
    target_names=target_columns,
    zero_division=0
))

print("Micro F1:",
      f1_score(y_val, y_pred_lstm, average='micro'))

print("Macro F1:",
      f1_score(y_val, y_pred_lstm, average='macro'))

**GRU Model**

In [ ]:
gru_model = Sequential([
    Embedding(
        input_dim=max_words,
        output_dim=128,
        mask_zero=True
    ),

    GRU(
        128,
        return_sequences=False
    ),

    Dropout(0.3),

    Dense(64, activation='relu'),

    Dropout(0.3),

    Dense(6, activation='sigmoid')
])

gru_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['binary_accuracy']
)

gru_model.summary()

In [ ]:
gru_callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True
    ),

    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2
    ),

    ModelCheckpoint(
        'best_gru.keras',
        monitor='val_loss',
        save_best_only=True
    )
]

history_gru = gru_model.fit(
    X_train_pad,
    y_train,
    validation_data=(X_val_pad, y_val),
    epochs=10,
    batch_size=128,
    callbacks=gru_callbacks
)

In [ ]:
y_prob_gru = gru_model.predict(X_val_pad)

y_pred_gru = (y_prob_gru >= 0.5).astype(int)

print(classification_report(
    y_val,
    y_pred_gru,
    target_names=target_columns,
    zero_division=0
))

print("Micro F1:",
      f1_score(y_val, y_pred_gru, average='micro'))

print("Macro F1:",
      f1_score(y_val, y_pred_gru, average='macro'))

**Bi-LSTM Model**

In [ ]:
bilstm_model = Sequential([
    Embedding(
        input_dim=max_words,
        output_dim=128,
        mask_zero=True
    ),

    Bidirectional(
        LSTM(64, return_sequences=False)
    ),

    Dropout(0.3),

    Dense(64, activation='relu'),

    Dropout(0.3),

    Dense(6, activation='sigmoid')
])

bilstm_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['binary_accuracy']
)

bilstm_model.summary()

In [ ]:
bilstm_callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True
    ),

    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2
    ),

    ModelCheckpoint(
        'best_bilstm.keras',
        monitor='val_loss',
        save_best_only=True
    )
]

history_bilstm = bilstm_model.fit(
    X_train_pad,
    y_train,
    validation_data=(X_val_pad, y_val),
    epochs=10,
    batch_size=128,
    callbacks=bilstm_callbacks
)

In [ ]:
y_prob_bilstm = bilstm_model.predict(X_val_pad)

y_pred_bilstm = (y_prob_bilstm >= 0.5).astype(int)

print(classification_report(
    y_val,
    y_pred_bilstm,
    target_names=target_columns,
    zero_division=0
))

print("Micro F1:",
      f1_score(y_val, y_pred_bilstm, average='micro'))

print("Macro F1:",
      f1_score(y_val, y_pred_bilstm, average='macro'))

**Comapring Models**

In [ ]:
results = []

model_predictions = {
    "Logistic Regression": y_pred,
    "LSTM": y_pred_lstm,
    "GRU": y_pred_gru,
    "BiLSTM": y_pred_bilstm
}

for name, predictions in model_predictions.items():

    micro = f1_score(
        y_val,
        predictions,
        average='micro'
    )

    macro = f1_score(
        y_val,
        predictions,
        average='macro'
    )

    results.append({
        'Model': name,
        'Micro F1': micro,
        'Macro F1': macro
    })

results_df = pd.DataFrame(results)

print(results_df)

In [ ]:
results_df.set_index('Model')[['Micro F1', 'Macro F1']].plot(
    kind='bar',
    figsize=(10, 5)
)

plt.title("Model Performance Comparison")
plt.ylabel("F1 Score")
plt.xticks(rotation=20)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import pickle

# Save tokenizer
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

# Save models
lstm_model.save("lstm_model.keras")
gru_model.save("gru_model.keras")
bilstm_model.save("bilstm_model.keras")

print("All models saved successfully!")

**Testing**

In [ ]:
def predict_comment(comment, model):

    sequence = tokenizer.texts_to_sequences([comment])

    padded = pad_sequences(
        sequence,
        maxlen=max_len,
        padding='post',
        truncating='post'
    )

    probabilities = model.predict(padded, verbose=0)[0]

    for label, probability in zip(target_columns, probabilities):

        status = "Yes" if probability >= 0.5 else "No"

        print(
            f"{label}: {status} "
            f"({probability:.4f})"
        )

In [ ]:
comment = "You are an idiot and nobody likes you"

predict_comment(comment, lstm_model)